# 01 — Exploratory Data Analysis
**Auction Marketplace Segmentation & Price Intelligence Engine**

This notebook explores the heavy equipment auction dataset: distributions, relationships, missing values, and business insights.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14

print("Libraries loaded")

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/listings_clean.csv', parse_dates=['list_date'])
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.dtypes

In [ ]:
print("=== Missing Values ===")
miss = df.isnull().sum()
print(miss[miss > 0])

In [ ]:
print("=== Descriptive Statistics ===")
df[['asking_price','hours_used','age_years','views','bids','inquiries','days_on_market']].describe().round(2)

## 2. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['asking_price'].plot.hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Asking Price ($)')
axes[0].set_title('Asking Price Distribution')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

np.log1p(df['asking_price']).plot.hist(bins=60, ax=axes[1], color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Log(Asking Price + 1)')
axes[1].set_title('Log Price Distribution')

plt.tight_layout()
plt.savefig('../reports/figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved price_distribution.png")

## 3. Category Analysis

In [ ]:
cat_stats = df.groupby('category').agg(
    count=('listing_id','count'),
    avg_price=('asking_price','mean'),
    median_price=('asking_price','median'),
    sold_rate=('sold','mean'),
).sort_values('count', ascending=False)
print(cat_stats.round(2))

fig, ax = plt.subplots(figsize=(12, 5))
cat_stats['avg_price'].sort_values().plot.barh(ax=ax, color='teal', alpha=0.8)
ax.set_xlabel('Average Asking Price ($)')
ax.set_title('Average Price by Equipment Category')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/category_avg_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Condition & Sold Rate

In [ ]:
cond_sold = df.groupby('condition')[['sold','asking_price']].agg({'sold':'mean','asking_price':'median'}).round(3)
print("Condition analysis:")
print(cond_sold)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cond_sold['sold'].sort_values().plot.barh(ax=axes[0], color='green', alpha=0.8)
axes[0].set_title('Sold Rate by Condition')
axes[0].set_xlabel('Sold Rate')

df.boxplot(column='asking_price', by='condition', ax=axes[1], grid=False)
axes[1].set_title('Price Distribution by Condition')
axes[1].set_xlabel('Condition')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/figures/condition_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Region Analysis

In [ ]:
region_stats = df.groupby('region').agg(
    count=('listing_id','count'),
    avg_price=('asking_price','mean'),
    sold_rate=('sold','mean'),
    avg_views=('views','mean'),
).sort_values('count', ascending=False)
print(region_stats.round(2))

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(region_stats)))
region_stats['count'].sort_values().plot.barh(ax=ax, color=colors, alpha=0.9)
ax.set_xlabel('Number of Listings')
ax.set_title('Listing Volume by Region')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/region_volume.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Engagement & Demand Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes, ['views','bids','inquiries'], ['steelblue','coral','green']):
    df[col].clip(upper=df[col].quantile(0.99)).plot.hist(bins=40, ax=ax, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(f'{col.capitalize()} Distribution')
    ax.set_xlabel(col.capitalize())
plt.tight_layout()
plt.savefig('../reports/figures/engagement_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['asking_price','age_years','hours_used','views','bids','inquiries','days_on_market','sold']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Seller Analysis

In [ ]:
seller_vol = df.groupby('seller_id').agg(
    listings=('listing_id','count'),
    sold_rate=('sold','mean'),
    avg_price=('asking_price','mean'),
).sort_values('listings', ascending=False)

print(f"Total sellers: {len(seller_vol):,}")
print(f"\nTop 10 sellers by volume:\n{seller_vol.head(10).round(2)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
seller_vol['listings'].clip(upper=50).plot.hist(bins=30, ax=axes[0], color='purple', alpha=0.8)
axes[0].set_xlabel('Listings per Seller')
axes[0].set_title('Seller Listing Volume Distribution')

seller_vol['sold_rate'].plot.hist(bins=30, ax=axes[1], color='orange', alpha=0.8)
axes[1].set_xlabel('Sold Rate')
axes[1].set_title('Seller Sold Rate Distribution')

plt.tight_layout()
plt.savefig('../reports/figures/seller_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Takeaways

- **Price range**: wide spread from <$10K forklifts to >$200K cranes
- **Sold rate**: varies significantly by condition and engagement  
- **Engagement**: highly skewed — most listings get moderate views, a few get massive attention
- **Regional demand**: Southeast and Midwest dominate listing volume
- **Seller patterns**: most sellers have few listings; power sellers drive disproportionate volume

In [ ]:
print('EDA Complete. All figures saved to reports/figures/')